# RAG evaluation logging (JSONL)

**Purpose:** Lightweight **observability** — log each query with retrieval scores and a simple **routing tier** (like production LLMOps: LangFuse / LiteLLM concepts, without those deps).

**Prerequisite:** Run `synthetic_policy_rag_walkthrough.ipynb` once so `outputs/chroma_synthetic_policy` exists, or run the setup cell below.

**What you learn:** How to structure logs for later dashboards, regression tests, and governance reviews.

In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime, timezone
import chromadb

ROOT = Path('..')
LOG_DIR = ROOT / 'outputs' / 'eval_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / 'rag_eval.jsonl'

def ensure_collection():
    policy_path = ROOT / 'synthetic_data' / 'mini_auto_policy.md'
    text = policy_path.read_text(encoding='utf-8')
    parts = re.split(r'\n##\s+', text)
    chunks = [parts[0].strip()] + [f"## {p.strip()}" for p in parts[1:] if p.strip()]
    out = ROOT / 'outputs' / 'chroma_synthetic_policy'
    out.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(out))
    try:
        client.delete_collection('synthetic_policy')
    except Exception:
        pass
    col = client.create_collection('synthetic_policy', metadata={'hnsw:space': 'cosine'})
    col.add(documents=chunks, ids=[f'chunk_{i}' for i in range(len(chunks))])
    return col

col = ensure_collection()

In [ ]:
def tier_from_distances(dists, k=3):
    """Map mean top-k similarity to RELIABLE / UNCERTAIN / UNRELIABLE (cosine distance)."""
    sims = [1.0 / (1.0 + d) for d in dists[:k]]
    m = sum(sims) / len(sims)
    if m > 0.72:
        return 'RELIABLE', m
    if m >= 0.55:
        return 'UNCERTAIN', m
    return 'UNRELIABLE', m

def log_query(col, question: str, run_id: str = 'demo'):
    r = col.query(query_texts=[question], n_results=5, include=['documents', 'distances', 'ids'])
    dists = r['distances'][0]
    tier, score = tier_from_distances(dists)
    row = {
        'ts': datetime.now(timezone.utc).isoformat(),
        'run_id': run_id,
        'question': question,
        'top_doc_ids': r['ids'][0][:3],
        'top_distances': [round(d, 5) for d in dists[:3]],
        'retrieval_score': round(score, 4),
        'tier': tier,
    }
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
    return row

queries = [
    'What is the rental reimbursement daily limit?',
    'Does my policy cover trips to Mars?',
    'Collision deductible amount',
    'roadside towing miles',
]
for q in queries:
    row = log_query(col, q)
    print(row['tier'], row['retrieval_score'], '|', q[:50])

In [ ]:
import pandas as pd
df = pd.read_json(LOG_FILE, lines=True)
df.tail(10)

## What we learned

- **Out-of-domain questions** (e.g. Mars) should show **worse** retrieval scores / UNRELIABLE tier — good sanity check for monitoring.
- **JSONL** appends safely; load with pandas for quick QA or ship to BigQuery/Snowflake in a real job.
- **AmFam-style roles** often mention observability stacks; this notebook is the **minimal viable pattern** you can describe in interviews.